# 02 · Préparer, puis prédire : un arbre de décision

**Séance 4 · deux blocs (environ 20 et 30 minutes) en séance, 15 minutes en autonomie.**

Seconde moitié de la question directrice : **« à qui devrions-nous envoyer la prochaine campagne ? »** Nous allons construire un modèle qui, à partir de ce qu'on sait d'un client, estime s'il répondra. Il apprend sur la campagne passée (`Response`).

Deux choses avant de commencer :

- en séance 3, nous avons **constaté** les anomalies sans rien corriger ; aujourd'hui, les décisions sont **appliquées**, une par une, par une fonction partagée ;
- ce modèle est un **exercice rétrospectif** : il explique ce qui distingue les répondants de la campagne passée. Le fichier ne permet pas d'établir que toutes les variables étaient connues avant l'envoi (section 9 du notebook 01). Nous y reviendrons dans les limites.

**Bloc 1 — Préparer et séparer** : appliquer les règles, choisir la cible et les variables, séparer apprentissage et test.
**Bloc 2 — L'arbre de décision** : le construire, le lire, mesurer ce qu'il vaut, repérer le surapprentissage.

## Bloc 1 — Préparer et séparer

### 1. Charger

In [ ]:
# Première cellule de chaque notebook : récupérer le module du cours, puis charger le fichier brut.
import os
import urllib.request

URL_MODULE = "https://raw.githubusercontent.com/2501Pr0ject/Data4Sup2Pub/main/notebooks/preparation.py"
if not os.path.exists("preparation.py"):
    urllib.request.urlretrieve(URL_MODULE, "preparation.py")

import pandas as pd
import matplotlib.pyplot as plt
import preparation as prep

brut = prep.charger()

### 2. Appliquer les décisions de nettoyage

**Question marketing :** *sur quels clients, exactement, allons-nous raisonner ?*

**La notion.** Chaque anomalie repérée en séance 3 a reçu une décision (garder, supprimer, recoder, signaler). Ces décisions sont écrites **une fois pour toutes** dans la fonction `preparer()` du module du cours, qui les applique dans l'ordre et annonce, pour chacune, le nombre de lignes concernées. Le tableau brut n'est jamais modifié : on obtient un **nouveau tableau**, `df`. Tous les notebooks suivants partent du même `df`.

In [ ]:
df = prep.preparer(brut)          # applique les règles et les annonce une par une
print()
print("Brut :", brut.shape, "| Préparé :", df.shape)

In [ ]:
# Les variables créées par la préparation (les colonnes d'origine sont conservées)
nouvelles = ["Age", "Anciennete_mois", "Enfants", "En_couple", "Depenses_total", "Achats_total",
             "Part_web", "Campagnes_acceptees", "Education_ord"]
df[nouvelles].describe().round(2).T[["min", "50%", "mean", "max"]]

**Question 1.** Quelle règle retire le plus de lignes ? Combien de clients reste-t-il, et quelle part du fichier a-t-on perdue ? Parmi les règles, laquelle auriez-vous tranchée autrement, et avec quelle conséquence ?

### 3. Choisir la cible et les variables

**Question marketing :** *que cherche-t-on à prédire, et avec quelles informations ?*

**La notion.** La **cible** est la colonne à prédire : `Response` (1 si le client a accepté l'offre de la dernière campagne). Les **variables explicatives** sont les colonnes que le modèle a le droit d'utiliser. On en exclut l'identifiant, les colonnes constantes, les textes bruts (remplacés par `En_couple` et `Education_ord`) et bien sûr la cible elle-même. Le module range les variables en quatre familles, selon qu'on peut établir ou non qu'elles étaient connues avant la campagne (notebook 01, section 9).

In [ ]:
cible = prep.CIBLE
variables = prep.JEU_COMPLET
print("Cible :", cible, "| nombre de variables explicatives :", len(variables))

familles = {"profil (établie)": prep.VARIABLES_PROFIL,
            "campagnes passées (plausible)": prep.VARIABLES_CAMPAGNES,
            "comportement d'achat (non établie)": prep.VARIABLES_COMPORTEMENT,
            "récence (non établie, la plus suspecte)": prep.VARIABLES_RECENCE}
pd.set_option("display.max_colwidth", None)      # afficher les listes en entier
pd.DataFrame({"variables": [", ".join(v) for v in familles.values()]}, index=familles.keys())

**Question 2.** Pourquoi `ID`, `Marital_Status`, `Education`, `Z_CostContact` et `Z_Revenue` ne sont-ils pas dans la liste ? Pourquoi garde-t-on quand même les familles « non établies » pour cet exercice, et qu'est-ce que cela nous interdit de prétendre ?

### 4. Séparer apprentissage et test

**Question marketing :** *comment savoir ce que vaudrait le modèle sur des clients qu'il n'a jamais vus ?*

**La notion.** Si l'on mesure un modèle sur les clients qui ont servi à le construire, il paraît meilleur qu'il n'est : il peut avoir **appris par cœur**. On sépare donc le fichier : **70 % des clients pour apprendre** (apprentissage, *train*), **30 % mis de côté pour mesurer** (test). La séparation est faite au hasard, en gardant la même part de répondants des deux côtés (**stratifiée**), avec une **graine fixe** pour que tout le monde obtienne exactement la même séparation, aujourd'hui et en séance 5.

In [ ]:
train, test = prep.separer_train_test(df)       # 70 / 30, stratifié sur la cible, graine fixe (42)

X_train, y_train = train[variables], train[cible]   # X : les variables explicatives, y : la cible
X_test, y_test = test[variables], test[cible]

print(f"Apprentissage : {len(train)} clients, dont {y_train.mean():.1%} de répondants")
print(f"Test          : {len(test)} clients, dont {y_test.mean():.1%} de répondants")

**Question 3.** Pourquoi ne pas mesurer le modèle sur les 2 026 clients, puisqu'on les a ? Que garantit la stratification ? Que se passerait-il, en séance 5, si la graine n'était pas fixée ?

## Bloc 2 — L'arbre de décision

### 5. Construire et lire un arbre

**Question marketing :** *quelles questions poser, dans quel ordre, pour reconnaître un client qui répondra ?*

**La notion.** Un **arbre de décision** est une suite de questions oui/non sur les variables (« a-t-il accepté au moins une campagne passée ? », « sa récence dépasse-t-elle 21 jours ? »). Chaque question coupe les clients en deux groupes plus « purs » que le groupe de départ ; au bout du chemin, une **feuille** donne la part de répondants parmi les clients qui y arrivent. L'algorithme choisit lui-même les questions et les seuils qui séparent le mieux. La **profondeur** est le nombre maximal de questions posées à un client. L'arbre est entraîné par la fonction `entrainer_arbre()` du module du cours : même graine, même méthode, de sorte que la séance 5 retrouve exactement le même arbre.

**Formulaire.** Choisissez la profondeur et relancez la cellule. Commencez par 3.

In [ ]:
#@title Profondeur de l'arbre
profondeur = "3"  #@param ["2", "3", "4", "6", "10", "sans limite"]
max_depth = None if profondeur == "sans limite" else int(profondeur)

from sklearn.tree import plot_tree

arbre = prep.entrainer_arbre(X_train, y_train, profondeur=max_depth)   # même fonction en séance 5 : même arbre garanti

exactitude_train = arbre.score(X_train, y_train)     # part de clients bien classés
exactitude_test = arbre.score(X_test, y_test)
modele_idiot = 1 - y_test.mean()                      # « personne ne répond » : juste pour tous les non-répondants
print(f"Profondeur {profondeur} : {arbre.get_n_leaves()} feuilles")
print(f"Exactitude (accuracy) sur l'apprentissage : {exactitude_train:.1%}")
print(f"Exactitude (accuracy) sur le test         : {exactitude_test:.1%}   (modèle idiot « personne ne répond » : {modele_idiot:.1%})")

fig, ax = plt.subplots(figsize=(22, 9))
plot_tree(arbre, feature_names=list(variables), class_names=["non", "oui"], max_depth=3,
          proportion=True, impurity=False, rounded=True, filled=True, fontsize=9, ax=ax)
ax.set_title("Les trois premiers niveaux de l'arbre (dans chaque case : part des clients, part [non, oui])", loc="left")
plt.show()

Comment lire une case : la première ligne est la question (si la réponse est oui, on descend à gauche) ; `samples` est la part des clients d'apprentissage qui arrivent là ; `value` donne, parmi eux, la part de non-répondants puis de répondants ; `class` est la réponse majoritaire. Les cases colorées en orange penchent vers « non », en bleu vers « oui ».

**Question 4.** Quelle est la première question que pose l'arbre ? Suivez le chemin « a accepté exactement une campagne passée, puis récence ≤ 21,5 jours » : quelle part de répondants dans cette feuille ? Quelle variable l'arbre utilise-t-il en deuxième et troisième position, et qu'en a-t-on dit au notebook 01 ?

### 6. Le modèle idiot et l'exactitude (accuracy)

**Question marketing :** *ce modèle vaut-il mieux que de ne rien faire ?*

**La notion.** L'**exactitude (accuracy)** est la part de clients bien classés. Elle a un piège : si 85 % des clients ne répondent pas, un modèle qui répond « non » à tout le monde a 85 % d'exactitude (accuracy) sans rien avoir appris. C'est le **modèle idiot**, la référence à battre.

In [ ]:
predictions = arbre.predict(X_test)
print("Clients du jeu de test que l'arbre annonce « répondra » :", int(predictions.sum()), "sur", len(predictions))
print("Vrais répondants dans le jeu de test                    :", int(y_test.sum()))
print(f"Exactitude (accuracy) de l'arbre : {exactitude_test:.1%} | du modèle idiot : {modele_idiot:.1%}")

**Question 5.** Avec la profondeur 3, l'arbre fait à peine mieux que le modèle idiot. Est-il inutile pour autant ? Qu'est-ce que l'exactitude (accuracy) ne dit pas ? (Indice : combien de répondants l'arbre annonce-t-il, et combien y en a-t-il vraiment ?)

### 7. Surapprentissage : apprendre par cœur

**Question marketing :** *un arbre plus profond est-il un meilleur modèle ?*

**La notion.** Plus l'arbre est profond, plus il pose de questions, jusqu'à isoler chaque client d'apprentissage dans sa propre feuille : il **apprend par cœur**. Son exactitude (accuracy) sur l'apprentissage monte vers 100 %, mais sur le test elle baisse. C'est le **surapprentissage**. La bonne profondeur est celle où le test est le meilleur, à lisibilité égale.

In [ ]:
profondeurs = [1, 2, 3, 4, 5, 6, 8, 10, 15]
resultats = []
for p in profondeurs:
    a = prep.entrainer_arbre(X_train, y_train, profondeur=p)
    resultats.append({"profondeur": p, "feuilles": a.get_n_leaves(),
                      "exactitude (accuracy) apprentissage": a.score(X_train, y_train), "exactitude (accuracy) test": a.score(X_test, y_test)})
resultats = pd.DataFrame(resultats).set_index("profondeur")
display(resultats.round(3))

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(resultats.index, resultats["exactitude (accuracy) apprentissage"], marker="o", color="#3b6ea5", linewidth=2, label="apprentissage")
ax.plot(resultats.index, resultats["exactitude (accuracy) test"], marker="o", color="#c0392b", linewidth=2, label="test")
ax.axhline(modele_idiot, color="#7f7f7f", linestyle="--", linewidth=1.5, label=f"modèle idiot ({modele_idiot:.1%})")
ax.set_xlabel("profondeur de l'arbre")
ax.set_ylabel("exactitude (accuracy)")
ax.set_title("Plus l'arbre est profond, plus il apprend par cœur", loc="left")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", color="#e5e5e5")
ax.set_axisbelow(True)
plt.show()

**Question 6.** À quelle profondeur l'exactitude (accuracy) sur le test est-elle la meilleure ? Que devient l'exactitude (accuracy) sur l'apprentissage quand la profondeur augmente ? Quelle profondeur recommandez-vous à l'équipe marketing, et pour quelles raisons (il y en a deux) ?

### 8. Les variables que l'arbre utilise

**Question marketing :** *sur quoi repose la décision du modèle ?*

**La notion.** L'**importance** d'une variable mesure sa contribution aux coupures de l'arbre (le total fait 100 %). Une variable absente du graphique n'a servi à aucune question. L'importance dit ce que le modèle **utilise**, pas ce qui **cause** la réponse.

In [ ]:
importances = pd.Series(arbre.feature_importances_, index=variables)
importances = importances[importances > 0].sort_values()

fig, ax = plt.subplots(figsize=(8, 0.5 * len(importances) + 1.2))
ax.barh(importances.index, importances.values, color="#3b6ea5")
for i, v in enumerate(importances.values):
    ax.text(v + 0.01, i, f"{v:.0%}", va="center", fontsize=9)
ax.set_xlim(0, 1)
ax.set_xlabel("importance dans l'arbre")
ax.set_title(f"Variables utilisées par l'arbre de profondeur {profondeur}", loc="left")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

familles_inverse = {v: f.split(" (")[0] for f, lst in familles.items() for v in lst}
part_non_etablie = importances[[v for v in importances.index if familles_inverse[v] in ("comportement d'achat", "récence")]].sum()
print(f"Part de l'importance portée par des variables dont la chronologie n'est pas établie : {part_non_etablie:.0%}")

**Question 7.** Quelle variable domine ? Quelle part de la décision repose sur des variables dont on ne sait pas si elles étaient connues avant la campagne ? Si l'on vous demandait « peut-on utiliser cet arbre pour la prochaine campagne ? », que répondriez-vous en deux phrases ?

### 9. Synthèse métier

- Après application des décisions de la séance 3, on raisonne sur **2 026 clients** ; chaque règle est écrite, comptée, et rejouable.
- Un arbre de profondeur 3 reconnaît les répondants d'abord par leur **historique de campagnes**, puis par la récence et l'ancienneté. Il s'explique en huit feuilles.
- Son exactitude (accuracy) de 85,7 % dépasse à peine le modèle idiot (84,7 %), et pourtant il annonce 58 répondants là où le modèle idiot n'en annonce aucun : **l'exactitude (accuracy) ne suffit pas à juger un modèle**. On l'évaluera autrement en séance 5, sur ce même arbre, en regardant quelles erreurs il fait et ce que chacune coûte.
- Au-delà de la profondeur 3, l'arbre **apprend par cœur** : meilleur sur l'apprentissage, pire sur le test.

### 10. Limites

- **Exercice rétrospectif.** La récence et les dépenses ont peut-être été mesurées après la campagne ; l'arbre les utilise. Il décrit la campagne passée, il n'est pas validé pour cibler la prochaine. Ce qu'il faudrait : une extraction datée avant l'envoi, la date de la campagne, puis un test réel sur un échantillon de clients.
- **Une seule séparation apprentissage/test.** Les chiffres (85,7 %, 58 répondants annoncés) dépendent du tirage ; une autre graine donnerait d'autres valeurs, proches mais différentes.
- **Prédire n'est pas agir.** Même un modèle parfait dirait qui répondra *si on l'a contacté comme lors du test* ; il ne dit pas ce qu'une offre différente, ou un canal différent, produirait.

### En autonomie (15 minutes)

Répondez par écrit, sans relancer le notebook :

1. Un collègue entraîne un arbre « sans limite » et annonce 99,5 % d'exactitude (accuracy). Que lui demandez-vous ?
2. L'arbre pose en premier la question « a accepté au moins une campagne passée ? ». Cette variable était-elle disponible avant la campagne ? Et `Recency` ?
3. Le modèle idiot a 84,7 % d'exactitude (accuracy). Expliquez en une phrase pourquoi ce chiffre n'est pas une performance.
4. Vrai ou faux : « si l'exactitude (accuracy) sur le test est inférieure à celle sur l'apprentissage, le modèle est mauvais ». Justifiez.
5. En une phrase : qu'est-ce qui change, en séance 5, et qu'est-ce qui ne change pas ?